In [1]:
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
import joblib  # To save the trained model
from skimage.feature import hog
import glob
import matplotlib.pyplot as plt

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from xgboost import XGBClassifier
from tqdm import tqdm

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Feature extaction using ResNet-50

In [2]:
# Define image transformation (ResNet50 requires 224x224 input)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
# Specify the GPU device (change "cuda:1" to the desired GPU index)
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

# Load pre-trained ResNet50 model and remove last FC layer
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])  # Remove last layer
model = model.to(device)  # Move model to the selected GPU
model.eval()

def extract_features(image_path):
    """Extract ResNet50 features from an image."""
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)  # Move image tensor to the selected GPU

    with torch.no_grad():
        features = model(image)
    return features.squeeze().cpu().numpy().flatten()  # Move result to CPU and convert to NumPy

In [20]:
def get_statistics(GT, prediction, file_name):
    # Evaluate model
    accuracy = accuracy_score(GT, prediction)
    print(f"Model Accuracy: {accuracy * 100:.2f}%")

    # Multiclass classification report
    report = classification_report(GT, prediction, target_names=[f'Grade {i}' for i in range(6)])
    print(report)

    # Generate classification report as a dictionary
    report_dict = classification_report(GT, prediction, output_dict=True)

    # Convert to DataFrame
    report_df = pd.DataFrame(report_dict).transpose()

    # Define the output folder path
    output_folder = "Reports"

    # Create the directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Save the classification report to CSV in the output folder
    report_path = os.path.join(output_folder, f"{file_name}.csv")
    report_df.to_csv(report_path, index=True)

    print(f"Classification report saved to: {report_path}")


def sens_and_spec(GT, prediction):
    print("MAE:", mean_absolute_error(GT, prediction))
    print("MSE:", mean_squared_error(GT, prediction))

    # SSE = MSE * number of samples
    sse = mean_squared_error(GT, prediction) * len(GT)
    print("SSE (Sum of Squared Errors):", sse)

    print("R² Score:", r2_score(GT, prediction))

    # Get confusion matrix
    cm = confusion_matrix(GT, prediction)

    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=["Na", "No", "Mi", "Mo", "Se", "Ex"],
                yticklabels=["Na", "No", "Mi", "Mo", "Se", "Ex"])
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()

    # Number of classes
    num_classes = cm.shape[0]

    # Sensitivity and Specificity for each class
    sensitivity = np.diag(cm) / np.sum(cm, axis=1)  # TP / (TP + FN)
    specificity = np.diag(cm) / (np.sum(cm, axis=0) + np.sum(cm, axis=1) - np.diag(cm))  # TN / (TN + FP)

    # Print metrics for each class
    for i in range(num_classes):
        print(f"Class {i} - Sensitivity: {sensitivity[i]:.4f}, Specificity: {specificity[i]:.4f}")

    macro_sensitivity = np.mean(sensitivity)
    macro_specificity = np.mean(specificity)

    print(f"Macro-Average Sensitivity: {macro_sensitivity:.4f}")
    print(f"Macro-Average Specificity: {macro_specificity:.4f}")

    class_support = np.sum(cm, axis=1)  # Total samples per class (true labels)
    weighted_sensitivity = np.sum(sensitivity * class_support) / np.sum(class_support)
    weighted_specificity = np.sum(specificity * class_support) / np.sum(class_support)

    print(f"Weighted Sensitivity: {weighted_sensitivity:.4f}")
    print(f"Weighted Specificity: {weighted_specificity:.4f}")


## Load Train and Test Data from RS1

In [ ]:
# Define the dataset path (Modify this with your actual dataset path)
dataset_train_path = "/train"
dataset_test_path = "/test"

# Define class labels
classes = ["Na", "No", "Mi", "Mo", "Se", "Ex"]

In [ ]:
# Load dataset
X_train = []  # Features
y_train = []  # Labels


# Loop through all images
for label, class_name in enumerate(classes):
    class_path = os.path.join(dataset_train_path, class_name)
    image_list = os.listdir(class_path)
    print(len(image_list))
    
    for img_name in tqdm(image_list, desc=f"Processing {class_name}", leave=False):
        img_path = os.path.join(class_path, img_name)
        features = extract_features(img_path)
        X_train.append(features)
        y_train.append(label)

# Convert to NumPy arrays and save
X_train = np.array(X_train)
y_train = np.array(y_train)

print(y_train)
print(len(y_train))


In [ ]:
# Load dataset
X_test = []  # Features
y_test = []  # Labels
image_names = []


# Loop through all images
for label, class_name in enumerate(classes):
    class_path = os.path.join(dataset_test_path, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        image_name = os.path.basename(img_path)
        features = extract_features(img_path)
        X_test.append(features)
        y_test.append(label)
        image_names.append(image_name)

# Convert to NumPy arrays and save
X_test = np.array(X_test)
y_test = np.array(y_test)

print(y_test)
print(len(y_test))


## Train XGB

In [34]:
# Initialize XGBoost classifier
clf_xgb = XGBClassifier(n_estimators=100, max_depth=5, random_state=42, use_label_encoder=False, eval_metric='mlogloss')

# Dictionary to store evaluation results
evals_result = {}

# Train with evaluation set (without evals_result in fit)
clf_xgb.fit(X_train, y_train, 
            eval_set=[(X_train, y_train), (X_test, y_test)], 
            verbose=True)

# Retrieve evaluation results manually
evals_result = clf_xgb.evals_result()

# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(evals_result['validation_0']['mlogloss'], label='Train Loss')
plt.plot(evals_result['validation_1']['mlogloss'], label='Validation Loss')
plt.xlabel('Iterations')
plt.ylabel('Log Loss')
plt.title('XGBoost Loss')
plt.legend()
plt.show()

In [30]:
# Predict on test data
y_pred = clf_xgb.predict(X_test)

print(y_pred)

# Print results
for img_name, pred, test in zip(image_names, y_pred, y_test):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred} -> GT: Grade {test}")

In [31]:
## Class level prediction
# Print the root mean squared error of the predictions
xgb_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {xgb_rmse_test}")

get_statistics(y_test, y_pred, 'xgb_test_report')

sens_and_spec(y_test, y_pred)

In [32]:
## Class continuous prediction
y_prob = clf_xgb.predict_proba(X_test)  # Get class probabilities
classes = np.arange(y_prob.shape[1])  # Class indices (0, 1, 2, ...)

# Compute weighted sum to get a continuous class prediction
y_continuous = np.sum(y_prob * classes, axis=1)

# Print with 4 decimal places
print([f"{val:.4f}" for val in y_continuous])

y_continuous_round = [round(x) for x in y_continuous]
print(y_continuous_round)
print("****************************************")

# Print the root mean squared error of the predictions
xgb_rmse_test_cont = np.sqrt(mean_squared_error(y_test, y_continuous))
print(f"RMSE: {xgb_rmse_test_cont}")

get_statistics(y_test, y_continuous_round, 'xgb_test_cont_round_report')

sens_and_spec(y_test, y_continuous_round)

In [35]:
# Get predicted probabilities from XGBoost model
y_pred_proba = clf_xgb.predict_proba(X_test)  # PROBABILITIES are needed

# Binarize the updated labels for multi-class ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4, 5])
n_classes = y_test_bin.shape[1]  # Should be 5 (after merging)

class_names = ["Na", "No", "Mi", "Mo", "Se", "Ex"]

# Plot ROC curve for each class
plt.figure(figsize=(8, 6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])  # FIXED: use probabilities
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--')

# Labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGB ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [13]:
# Save the trained model
joblib.dump(clf_xgb, "xgb_cnv_classifier.pkl")

## Train Random Forest

In [ ]:
# Train Random Forest Classifier
# clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rf = RandomForestClassifier(n_estimators=200, max_depth=40, min_samples_split=2, min_samples_leaf=1, max_features='log2',  random_state=42)
clf_rf.fit(X_train, y_train)

In [ ]:
# Predict on test data
y_pred_rf = clf_rf.predict(X_test)
y_pred_rf

# Print results
for img_name, pred, test in zip(image_names, y_pred_rf, y_test):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred} -> GT: Grade {test}")

In [37]:
rf_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f"RMSE: {rf_rmse_test}")

get_statistics(y_test, y_pred_rf, 'rf_test_report')

sens_and_spec(y_test, y_pred_rf)

In [ ]:
## Class continuous prediction
y_prob = clf_rf.predict_proba(X_test)  # Get class probabilities
classes = np.arange(y_prob.shape[1])  # Class indices (0, 1, 2, ...)

# Compute weighted sum to get a continuous class prediction
y_continuous = np.sum(y_prob * classes, axis=1)

# Print with 4 decimal places
print([f"{val:.4f}" for val in y_continuous])

y_continuous_round = [round(x) for x in y_continuous]
print(y_continuous_round)
print("****************************************")

# Print the root mean squared error of the predictions
rf_rmse_test_cont = np.sqrt(mean_squared_error(y_test, y_continuous))
print(f"RMSE: {rf_rmse_test_cont}")

get_statistics(y_test, y_continuous_round, 'rf_test_cont_round_report')

sens_and_spec(y_test, y_continuous_round)

In [40]:
# Get predicted probabilities from XGBoost model
y_pred_proba = clf_rf.predict_proba(X_test)  # PROBABILITIES are needed

# Binarize the updated labels for multi-class ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4, 5])
n_classes = y_test_bin.shape[1]  # Should be 5 (after merging)

class_names = ["Na", "No", "Mi", "Mo", "Se", "Ex"]

# Plot ROC curve for each class
plt.figure(figsize=(8, 6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])  # FIXED: use probabilities
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--')

# Labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [19]:
# Save the trained model
joblib.dump(clf_rf, "random_forest_cnv_classifier.pkl")

## Prediction of Mask-RCNN Multi-Class

In [ ]:
validation_maskrcnn_pred = [0,	0,	0,	0,	0,	1,	5,	1,	5,	5,	2,	2,	3,	2,	2,	3,	3,	3,	3,	3,	4,	5,	3,	3,	3,	5,	5,	5,	5,	5]

# Print results
for img_name, pred, test in zip(image_names, validation_maskrcnn_pred, y_test):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred} -> GT: Grade {test}")
    

In [50]:
maskrcnn_rmse_test = np.sqrt(mean_squared_error(y_test, validation_maskrcnn_pred))
print(f"RMSE: {maskrcnn_rmse_test}")

get_statistics(y_test, validation_maskrcnn_pred, 'maskrcnn_test_report')

sens_and_spec(y_test, validation_maskrcnn_pred)

## Plot RS-1 RMSE Class Results

In [52]:
# Store RMSE values
methods = ["Mask-RCNN Multi-Class", "RF-CNN-Feature", "XGB-CNN-Feature"]
rmse_values = [maskrcnn_rmse_test, rf_rmse_test, xgb_rmse_test]

plt.figure(figsize=(8, 5))
bars = plt.bar(methods, rmse_values, color=['blue', 'green', 'red'])

# Annotate each bar with RMSE value
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.4f}', 
             ha='center', fontsize=10, fontweight='bold')

plt.ylabel("RMSE")
plt.title("RMSE Comparison Across Methods")
plt.ylim(0, max(rmse_values) + 0.3)  # Adjust y-axis for better visibility
plt.show()

In [42]:
# Store RMSE values
methods = ["RF-CNN-Feature", "XGB-CNN-Feature"]
rmse_values = [rf_rmse_test_cont, xgb_rmse_test_cont]

plt.figure(figsize=(6, 5))
bars = plt.bar(methods, rmse_values, color=['green', 'red'])

# Annotate each bar with RMSE value
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.4f}', 
             ha='center', fontsize=10, fontweight='bold')

plt.ylabel("RMSE")
plt.title("RMSE Comparison Across Methods")
plt.ylim(0, max(rmse_values) + 0.3)  # Adjust y-axis for better visibility
plt.show()

## Test on Unseen 24 Images

In [ ]:
# Define the dataset path (Modify this with your actual dataset path)
dataset_unseen_path = "/CNV-New-Data-Paper"
# Define class labels
classes = ["Na", "No", "Mi", "Mo", "Se", "Ex"]

# Load dataset
X_unseen = []  # Features
y_unseen = []  # Labels
image_names = []

# Loop through all images
for label, class_name in enumerate(classes):
    class_path = os.path.join(dataset_unseen_path, class_name)
    image_list = os.listdir(class_path)
    print(len(image_list))
    
    for img_name in tqdm(image_list, desc=f"Processing {class_name}", leave=False):
        img_path = os.path.join(class_path, img_name)
        features = extract_features(img_path)
        X_unseen.append(features)
        y_unseen.append(label)
        image_names.append(img_name)

# Convert to NumPy arrays and save
X_unseen = np.array(X_unseen)
y_unseen = np.array(y_unseen)

print(y_unseen)
print(len(y_unseen))

## Test on Unseen Images using XGB

In [54]:
clf_xgb = joblib.load('xgb_cnv_classifier.pkl')


# Predict on test data
y_pred = clf_xgb.predict(X_unseen)

print(y_pred)

# Print results
for img_name, pred, gt_unseen in zip(image_names, y_pred, y_unseen):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred}  -> GT: Grade {gt_unseen}")

In [65]:
for img_name, pred, gt_unseen in zip(image_names, y_pred, y_unseen):
    print(f"{img_name}")

In [55]:
xgb_rmse_unseen = np.sqrt(mean_squared_error(y_unseen, y_pred))
print(f"RMSE: {xgb_rmse_unseen}")

get_statistics(y_unseen, y_pred, 'xgb_unseen_report')

sens_and_spec(y_unseen, y_pred)

In [56]:
## Class continuous prediction
y_prob = clf_xgb.predict_proba(X_unseen)  # Get class probabilities
classes = np.arange(y_prob.shape[1])  # Class indices (0, 1, 2, ...)

# Compute weighted sum to get a continuous class prediction
y_continuous = np.sum(y_prob * classes, axis=1)

# Print with 4 decimal places
print([f"{val:.4f}" for val in y_continuous])

y_continuous_round = [round(x) for x in y_continuous]
print(y_continuous_round)
print("****************************************")

# Print the root mean squared error of the predictions
xgb_rmse_unseen_cont = np.sqrt(mean_squared_error(y_unseen, y_continuous))
print(f"RMSE: {xgb_rmse_unseen_cont}")

get_statistics(y_unseen, y_continuous_round, 'xgb_unseen_cont_round_report')

sens_and_spec(y_unseen, y_continuous_round)

In [58]:
# Get predicted probabilities from XGBoost model
y_pred_proba = clf_xgb.predict_proba(X_unseen)  # PROBABILITIES are needed

# Binarize the updated labels for multi-class ROC
y_test_bin = label_binarize(y_unseen, classes=[0, 1, 2, 3, 4, 5])
n_classes = y_test_bin.shape[1]  # Should be 5 (after merging)

# Plot ROC curve for each class
plt.figure(figsize=(8, 6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])  # FIXED: use probabilities
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--')

# Labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC Curve')
plt.legend(loc='lower right')
plt.show()

## Test on Unseen using Random Forest

In [59]:
clf_rf = joblib.load('random_forest_cnv_classifier.pkl')

# Predict on test data
y_pred = clf_rf.predict(X_unseen)

print(y_pred)

# Print results
for img_name, pred in zip(image_names, y_pred):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred-1}")

In [60]:
rf_rmse_unseen = np.sqrt(mean_squared_error(y_unseen, y_pred))
print(f"RMSE: {rf_rmse_unseen}")

get_statistics(y_unseen, y_pred, 'rf_unseen_report')

sens_and_spec(y_unseen, y_pred)

In [62]:
## Class continuous prediction
y_prob = clf_rf.predict_proba(X_unseen)  # Get class probabilities
classes = np.arange(y_prob.shape[1])  # Class indices (0, 1, 2, ...)

# Compute weighted sum to get a continuous class prediction
y_continuous = np.sum(y_prob * classes, axis=1)

# Print with 4 decimal places
print([f"{val:.4f}" for val in y_continuous])

y_continuous_round = [round(x) for x in y_continuous]
print(y_continuous_round)
print("****************************************")

# Print the root mean squared error of the predictions
rf_rmse_unseen_cont = np.sqrt(mean_squared_error(y_unseen, y_continuous))
print(f"RMSE: {rf_rmse_unseen_cont}")

get_statistics(y_unseen, y_continuous_round, 'rf_unseen_cont_round_report')

sens_and_spec(y_unseen, y_continuous_round)

In [64]:
# Get predicted probabilities from XGBoost model
y_pred_proba = clf_rf.predict_proba(X_unseen)  # PROBABILITIES are needed

# Binarize the updated labels for multi-class ROC
y_test_bin = label_binarize(y_unseen, classes=[0, 1, 2, 3, 4, 5])
n_classes = y_test_bin.shape[1]  # Should be 5 (after merging)

# Plot ROC curve for each class
plt.figure(figsize=(8, 6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])  # FIXED: use probabilities
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--')

# Labels and title
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [66]:
test_maskrcnn_pred = [0,	0,	0,	0,	0,	1,	1,	3,	3,	1,	2,	1,	2,	1,	2,	3,	4,	3,	3,	5,	4,	5,	5,	5,	4,	5,	5,	5,	3,	5]

# Print results
for img_name, pred, test in zip(image_names, test_maskrcnn_pred, y_unseen):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred} -> GT: Grade {test}")

maskrcnn_rmse_unseen = np.sqrt(mean_squared_error(y_unseen, test_maskrcnn_pred))
print(f"RMSE: {maskrcnn_rmse_unseen}")

get_statistics(y_unseen, test_maskrcnn_pred, 'maskrcnn_unseen_report')

sens_and_spec(y_unseen, test_maskrcnn_pred)

In [67]:
# Store RMSE values
methods = ["Mask-RCNN Multi-Class", "RF-CNN-Feature", "XGB-CNN-Feature"]
rmse_values = [maskrcnn_rmse_unseen, rf_rmse_unseen, xgb_rmse_unseen]

plt.figure(figsize=(8, 5))
bars = plt.bar(methods, rmse_values, color=['blue', 'green', 'red'])

# Annotate each bar with RMSE value
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.4f}', 
             ha='center', fontsize=10, fontweight='bold')

plt.ylabel("RMSE")
plt.title("RMSE Comparison Across Methods")
plt.ylim(0, max(rmse_values) + 0.3)  # Adjust y-axis for better visibility
plt.show()

In [32]:
# Store RMSE values
methods = ["RF-CNN-Feature", "XGB-CNN-Feature"]
rmse_values = [rf_rmse_unseen_cont, xgb_rmse_unseen_cont]

plt.figure(figsize=(6, 5))
bars = plt.bar(methods, rmse_values, color=['green', 'red'])

# Annotate each bar with RMSE value
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.4f}', 
             ha='center', fontsize=10, fontweight='bold')

plt.ylabel("RMSE")
plt.title("RMSE Comparison Across Methods")
plt.ylim(0, max(rmse_values) + 0.3)  # Adjust y-axis for better visibility
plt.show()

In [6]:
import os
from tqdm import tqdm
import numpy as np

# Define the dataset path
dataset_unseen_path = "/usr/mvl2/grzc7/CNV-MaskRCNN/MaskRCNN-UNET-Pytorch/DeepCNV-Paper/Figure-4"

# Define class labels
classes = ["Na", "No", "Mi", "Mo", "Se", "Ex"]

# Initialize storage
X_unseen = []
y_unseen = []
image_names = []

# Loop through all class folders
for label, class_name in enumerate(classes):
    class_path = os.path.join(dataset_unseen_path, class_name)
    
    # Skip if folder does not exist
    if not os.path.isdir(class_path):
        print(f"Skipping missing folder: {class_path}")
        continue

    image_list = os.listdir(class_path)
    print(f"{class_name} - {len(image_list)} images")

    for img_name in tqdm(image_list, desc=f"Processing {class_name}", leave=False):
        img_path = os.path.join(class_path, img_name)
        features = extract_features(img_path)
        X_unseen.append(features)
        y_unseen.append(label)
        image_names.append(img_name)

# Convert to NumPy arrays
X_unseen = np.array(X_unseen)
y_unseen = np.array(y_unseen)

print("Labels:", y_unseen)
print("Total images processed:", len(y_unseen))


In [18]:
clf_xgb = joblib.load('xgb_cnv_classifier.pkl')


# Predict on test data
y_pred = clf_xgb.predict(X_unseen)

print(y_pred)

# Print results
for img_name, pred, gt_unseen in zip(image_names, y_pred, y_unseen):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred}  -> GT: Grade {gt_unseen}")

In [19]:
## Class continuous prediction
y_prob = clf_xgb.predict_proba(X_unseen)  # Get class probabilities
classes = np.arange(y_prob.shape[1])  # Class indices (0, 1, 2, ...)

# Compute weighted sum to get a continuous class prediction
y_continuous = np.sum(y_prob * classes, axis=1)

# Print with 4 decimal places
print([f"{val:.4f}" for val in y_continuous])

y_continuous_round = [round(x) for x in y_continuous]
print(y_continuous_round)

# Print results
for img_name, pred, gt_unseen in zip(image_names, y_continuous, y_unseen):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred-1}  -> GT: Grade {gt_unseen-1}")

print("****************************************")

# Print the root mean squared error of the predictions
xgb_rmse_unseen_cont = np.sqrt(mean_squared_error(y_unseen, y_continuous))
print(f"RMSE: {xgb_rmse_unseen_cont}")

In [20]:
clf_rf = joblib.load('random_forest_cnv_classifier.pkl')

# Predict on test data
y_pred = clf_rf.predict(X_unseen)

print(y_pred)

# Print results
for img_name, pred in zip(image_names, y_pred):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred}")

In [21]:
## Class continuous prediction
y_prob = clf_rf.predict_proba(X_unseen)  # Get class probabilities
classes = np.arange(y_prob.shape[1])  # Class indices (0, 1, 2, ...)

# Compute weighted sum to get a continuous class prediction
y_continuous = np.sum(y_prob * classes, axis=1)

# Print with 4 decimal places
print([f"{val:.4f}" for val in y_continuous])

y_continuous_round = [round(x) for x in y_continuous]
print(y_continuous_round)

# Print results
for img_name, pred, gt_unseen in zip(image_names, y_continuous, y_unseen):
    print(f"Image: {img_name} -> Predicted Grade: Grade {pred-1}  -> GT: Grade {gt_unseen-1}")

print("****************************************")

# Print the root mean squared error of the predictions
rf_rmse_unseen_cont = np.sqrt(mean_squared_error(y_unseen, y_continuous))
print(f"RMSE: {rf_rmse_unseen_cont}")